# Experiment 8 — Intelligent Customer Support System with Escalation
### Case Study: SkyBridge Airways Passenger Support Desk (Aviation / Travel Domain)

**Scenario.** *SkyBridge Airways* (fictional) answers thousands of passenger messages. Questions about baggage allowance
or web check-in can be answered from approved policy text, but a passenger describing **chest pain on board**, a **child
travelling alone**, or a **cancelled connection** must reach a person immediately. The desk therefore:

```
queries.csv ─► ingestion + sanitizing ─► classification (intent · complexity · priority) ─► knowledge-base match
      ─► workflow router ─┬─ P1 safety ............. immediate human escalation (15 min SLA)
                          ├─ match >= 70, low risk .. automatic draft answer
                          ├─ match 40-69.99 ......... agent review
                          └─ match < 40 ............. escalate + request information
   ticket lifecycle (SQLite) · human actions · notifications with retry · SLA monitor · hash-chained audit log
```

Everything is synthetic (fictional airline, invented policies, invented passengers) and runs offline with no API key.
Run all cells from top to bottom.

## 1. Setup and configuration (`config.json`)

In [ ]:
import csv, hashlib, html, json, logging, re, sqlite3, time
from collections import Counter
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

WORK = Path("airline_support")
WORK.mkdir(exist_ok=True)
for old in WORK.rglob("*"):                          # every run starts from a clean state
    if old.is_file():
        old.unlink()
logging.basicConfig(filename=WORK / "support.log", level=logging.INFO, force=True,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("airline-support")

CONFIG = {
    "sla_minutes": {"P1": 15, "P2": 60, "P3": 240, "P4": 1440},      # first-response deadlines
    "at_risk_fraction": 0.8,                                          # AT_RISK once 80 % of the SLA has elapsed
    "thresholds": {"auto_resolve": 70, "agent_review": 40},           # knowledge-base match score (0-100)
    "retry": {"max_attempts": 3, "base_delay_s": 1.0, "factor": 2},   # waits 1 s, then 2 s, ... between attempts
    "reviewers": {"AGT_01": "agent", "AGT_02": "agent", "SUP_01": "supervisor", "OBS_01": "viewer"},
}
(WORK / "config.json").write_text(json.dumps(CONFIG, indent=2))
INTERACTIVE_HITL = False        # True: the supervisor console in section 9 asks for real keyboard input
NOW = datetime(2026, 11, 10, 10, 0)    # simulated "current time" used by the SLA board
CONSOLE_TIME = datetime(2026, 11, 10, 9, 41)   # simulated time at which the supervisor opens the console
print("config.json written | SLA minutes:", CONFIG["sla_minutes"], "| reviewers:", len(CONFIG["reviewers"]))

## 2. Approved knowledge base (`knowledge_base.json`)

Only these reviewed articles may be used to answer passengers. Loading is defensive: a malformed file is logged and the desk
falls back to *no knowledge*, which escalates every query to a person instead of guessing.

In [ ]:
KB_ARTICLES = [
    {"id": "KB01", "title": "Checked baggage allowance", "category": "GENERAL_INFO",
     "keywords": ["baggage", "bag", "bags", "luggage", "allowance", "kg", "checked", "weight", "limit", "economy", "class", "free"],
     "answer": "Economy fares include one checked bag of up to 20 kg and Business fares include two bags of up to 32 kg each. Extra weight is charged at the airport counter rate."},
    {"id": "KB02", "title": "Web check-in", "category": "CHECKIN_SERVICES",
     "keywords": ["web", "online", "check", "boarding", "pass", "open", "opens", "close", "closes", "hours", "flight", "seat"],
     "answer": "Web check-in opens 48 hours before departure and closes 2 hours before departure. Your boarding pass is available in the app and by e-mail after check-in."},
    {"id": "KB03", "title": "Refund for a cancelled flight", "category": "REFUND_CHANGES",
     "keywords": ["refund", "refunded", "cancelled", "cancellation", "flight", "booked", "card", "payment", "original", "money", "days", "process"],
     "answer": "If SkyBridge cancels your flight you may request a full refund to the original payment method. Refunds are processed within 7 working days of the request."},
    {"id": "KB04", "title": "Change of travel date and change fee", "category": "REFUND_CHANGES",
     "keywords": ["change", "travel", "date", "fee", "reschedule", "booking", "fare", "difference", "next", "move"],
     "answer": "Date changes made at least 24 hours before departure carry a change fee that depends on the fare family, plus any fare difference."},
    {"id": "KB05", "title": "Name correction", "category": "REFUND_CHANGES",
     "keywords": ["name", "correction", "spelling", "spelt", "passport", "typo", "mismatch", "ticket", "wrongly", "surname"],
     "answer": "A spelling correction of up to three characters is free once per booking if requested at least 24 hours before departure. A change to a different name is not permitted."},
    {"id": "KB06", "title": "Delayed, lost or damaged baggage", "category": "BAGGAGE_ISSUE",
     "keywords": ["lost", "delayed", "missing", "damaged", "baggage", "bag", "luggage", "arrive", "tag", "report", "desk", "claim", "trace"],
     "answer": "Report the problem at the baggage service desk within 24 hours of arrival (7 days for damage) and keep your baggage tag. We trace the bag and deliver it to your address, normally within 48 hours."},
    {"id": "KB07", "title": "Flight disruption and rebooking", "category": "FLIGHT_DISRUPTION",
     "keywords": ["delayed", "cancelled", "missed", "miss", "connection", "rebook", "rebooked", "rebooking", "alternative", "flight", "today"],
     "answer": "When SkyBridge disrupts a flight we rebook you free of charge on the next available flight. For delays of more than 3 hours we provide meal vouchers at the airport."},
    {"id": "KB08", "title": "Special assistance and special meals", "category": "CHECKIN_SERVICES",
     "keywords": ["wheelchair", "assistance", "special", "meal", "meals", "mobility", "vegetarian", "diabetic", "request", "arrange"],
     "answer": "Wheelchair assistance and special meals must be requested at least 48 hours before departure through Manage Booking or the call centre."},
    {"id": "KB09", "title": "Frequent flyer miles", "category": "GENERAL_INFO",
     "keywords": ["miles", "mile", "points", "loyalty", "flyer", "credit", "status", "tier", "earn", "earned", "missing"],
     "answer": "Miles are credited within 7 days of travel. Missing miles can be claimed within 90 days by sending the boarding pass."},
    {"id": "KB10", "title": "Airport arrival time", "category": "GENERAL_INFO",
     "keywords": ["airport", "arrive", "reach", "early", "counter", "counters", "close", "time", "international", "domestic", "hours"],
     "answer": "Check-in counters close 60 minutes before departure. Please reach the airport 2 hours before a domestic flight and 3 hours before an international flight."},
]
for a in KB_ARTICLES:
    a.update({"approved_by": "Customer Care Policy Board", "last_reviewed": "2026-09-30"})
(WORK / "knowledge_base.json").write_text(json.dumps({"articles": KB_ARTICLES}, indent=2))


def load_kb(path):
    """Defensive loader -> (articles, error). A bad file never crashes the desk."""
    try:
        articles = json.loads(Path(path).read_text(encoding="utf-8"))["articles"]
        for a in articles:
            assert {"id", "title", "keywords", "answer"} <= set(a)
        return articles, None
    except (OSError, ValueError, KeyError, TypeError, AssertionError) as exc:
        log.error("knowledge base unavailable: %s", exc)
        return [], f"knowledge base unavailable: {type(exc).__name__}"


kb, kb_error = load_kb(WORK / "knowledge_base.json")
print(f"knowledge_base.json: {len(kb)} approved articles, error = {kb_error}")

## 3. Synthetic passenger queries (`customer_queries.csv`)

Twenty rows: normal queries from every category plus edge cases: two empty queries, a duplicate query ID, and a message with
a `<script>` tag, a phone number and a booking reference. Ground-truth labels are kept **separately** for the accuracy metric.

In [ ]:
Q = lambda qid, cust, text, created, email=None: {"query_id": qid, "customer_id": cust, "customer_email": email or f"{cust.lower()}@mail.example",
                                                  "query_text": text, "created_at": created}
rows = [
    Q("Q001", "C101", "What is the checked baggage allowance for economy class?", "2026-11-10T09:50:00"),
    Q("Q002", "C102", "When does web check-in open for my flight SB204?", "2026-11-10T09:30:00"),
    Q("Q003", "C103", "I need a refund for my cancelled flight SB118, booked with card", "2026-11-10T09:00:00"),
    Q("Q004", "C104", "My flight SB330 got cancelled and I will miss my connection, I need to be rebooked today", "2026-11-10T09:10:00"),
    Q("Q005", "C105", "I am feeling chest pain and I am on flight SB512 right now", "2026-11-10T09:40:00"),
    Q("Q006", "C106", "My baggage did not arrive at Delhi, tag number SB908812. Please help.", "2026-11-10T08:45:00", "c106@flaky.example"),
    Q("Q007", "C107", "Can you change my travel date for the booking to next Friday and what is the fee?", "2026-11-10T06:30:00"),
    Q("Q008", "C108", "I have contacted you three times about my lost bag and nobody has replied. This is unacceptable.", "2026-11-10T09:20:00", "c108@invalid.test"),
    Q("Q009", "C109", "Please arrange wheelchair assistance for my mother", "2026-11-10T07:40:00"),
    Q("Q010", "C110", "How many miles will I earn for flight SB204?", "2026-11-10T08:05:00"),
    Q("Q011", "C111", "", "2026-11-10T08:10:00"),
    Q("Q012", "C112", "    ", "2026-11-10T08:12:00"),
    Q("Q003", "C113", "Where is my refund?", "2026-11-10T08:15:00"),
    Q("Q014", "C113", "<script>alert('x')</script> What time do check-in counters close? My number is 9876543210 and PNR K7P2QZ", "2026-11-10T07:55:00"),
    Q("Q015", "C114", "I was denied boarding even though I had a confirmed ticket and I was told to wait", "2026-11-10T08:20:00"),
    Q("Q016", "C115", "The name on my ticket is spelt wrongly, my passport says Sharma but the ticket says Sharmaa", "2026-11-10T08:50:00"),
    Q("Q017", "C116", "My 8 year old son is travelling alone on SB611 and we cannot find the airline staff at the gate", "2026-11-10T09:30:00"),
    Q("Q018", "C117", "How early should I reach the airport for an international flight?", "2026-11-10T07:20:00"),
    Q("Q019", "C118", "I want compensation and will take legal action, my flight was delayed 6 hours and nobody helped", "2026-11-10T08:30:00"),
    Q("Q020", "C119", "Can I get a vegetarian meal for flight SB204?", "2026-11-10T09:45:00"),
]
pd.DataFrame(rows).to_csv(WORK / "customer_queries.csv", index=False)

GROUND_TRUTH = {   # hand-labelled intent and priority for the 17 valid queries (never shown to the classifier)
    "Q001": ("GENERAL_INFO", "P4"), "Q002": ("CHECKIN_SERVICES", "P3"), "Q003": ("REFUND_CHANGES", "P3"),
    "Q004": ("FLIGHT_DISRUPTION", "P2"), "Q005": ("SAFETY_EMERGENCY", "P1"), "Q006": ("BAGGAGE_ISSUE", "P3"),
    "Q007": ("REFUND_CHANGES", "P3"), "Q008": ("UNRESOLVED_GRIEVANCE", "P2"), "Q009": ("CHECKIN_SERVICES", "P3"),
    "Q010": ("GENERAL_INFO", "P4"), "Q014": ("GENERAL_INFO", "P4"), "Q015": ("FLIGHT_DISRUPTION", "P2"),
    "Q016": ("REFUND_CHANGES", "P3"), "Q017": ("SAFETY_EMERGENCY", "P1"), "Q018": ("GENERAL_INFO", "P4"),
    "Q019": ("FLIGHT_DISRUPTION", "P2"), "Q020": ("CHECKIN_SERVICES", "P3")}
print(f"customer_queries.csv written: {len(rows)} rows | ground-truth labels: {len(GROUND_TRUTH)}")

## 4. Component 1 — Query ingestion and input sanitizing

Required fields are checked, empty queries and duplicate IDs are rejected, HTML is stripped, and personal data (e-mail,
phone, booking reference, card number) is **masked before the text is stored, classified or logged**.

In [ ]:
REQUIRED_COLS = ["query_id", "customer_id", "query_text", "created_at"]
SCRIPT_RE, TAG_RE = re.compile(r"<script.*?>.*?</script>", re.I | re.S), re.compile(r"<[^>]+>")
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
CARD_RE = re.compile(r"(?<!\d)(?:\d[ -]?){13,16}(?!\d)")
PHONE_RE = re.compile(r"(?<!\d)(?:\+?91[\s-]?)?[6-9]\d{9}(?!\d)")
PNR_RE = re.compile(r"\b(?=[A-Z0-9]{6}\b)(?=[A-Z0-9]*\d)(?=[A-Z0-9]*[A-Z])[A-Z0-9]{6}\b")


def clean_input(text):
    text = SCRIPT_RE.sub(" ", html.unescape(text or ""))
    text = TAG_RE.sub(" ", text)
    return re.sub(r"\s+", " ", re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)).strip()


def mask_pii(text):
    text = EMAIL_RE.sub("[redacted-email]", text)
    for pattern in (CARD_RE, PHONE_RE, PNR_RE):
        text = pattern.sub("[redacted-number]", text)
    return text


def load_queries(path):
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        missing = [c for c in REQUIRED_COLS if c not in (reader.fieldnames or [])]
        if missing:
            raise ValueError(f"required columns missing: {missing}")
        return list(reader)


def ingest(raw_rows):
    """-> (valid_records, rejected). A record carries the sanitized, PII-masked text."""
    valid, rejected, seen = [], [], set()
    for row in raw_rows:
        qid = (row.get("query_id") or "").strip()
        text = mask_pii(clean_input(row.get("query_text")))
        try:
            created = datetime.fromisoformat((row.get("created_at") or "").strip())
        except ValueError:
            created = None
        if not qid or not (row.get("customer_id") or "").strip() or created is None:
            rejected.append({"query_id": qid or "(blank)", "category": "MISSING_FIELD", "reason": "query_id, customer_id or created_at missing/invalid"})
        elif not text:
            rejected.append({"query_id": qid, "category": "EMPTY_QUERY", "reason": "query_text is empty after cleaning"})
        elif qid in seen:
            rejected.append({"query_id": qid, "category": "DUPLICATE_ID", "reason": "query_id already ingested; later row ignored"})
        else:
            seen.add(qid)
            valid.append({"query_id": qid, "customer_id": row["customer_id"].strip(), "email": (row.get("customer_email") or "").strip(),
                          "text": text, "created_at": created})
    return valid, rejected


raw_rows = load_queries(WORK / "customer_queries.csv")
valid_preview, rejected_preview = ingest(raw_rows)
print(f"INGESTION  customer_queries.csv: {len(raw_rows)} rows read | {len(valid_preview)} valid | {len(rejected_preview)} rejected\n")
for r in rejected_preview:
    print(f"  {r['query_id']:<6}{r['category']:<14}{r['reason']}")
raw14 = next(r for r in raw_rows if r["query_id"] == "Q014")["query_text"]
stored14 = next(r for r in valid_preview if r["query_id"] == "Q014")["text"]
print("\nSANITIZING  Q014 (HTML removed, personal data masked before storage)")
print(f"  raw    : {raw14}")
print(f"  stored : {stored14}")
import textwrap
print("\n" + textwrap.fill(f"KNOWLEDGE BASE  {len(kb)} approved articles: " + "; ".join(f"{a['id']} {a['title']}" for a in kb), width=100, subsequent_indent="  "))

## 5. Component 2 — Classification engine (intent, complexity, priority)

Weighted regular-expression rules score each of seven intents; the highest score wins (ties go to the earlier, safer intent).
**Safety triggers override everything**: a P1 priority is assigned from the intent, never from a knowledge-base match.

In [ ]:
INTENT_RULES = {   # (pattern, weight); intents are listed from safest to most routine and win ties in that order
    "SAFETY_EMERGENCY": [(r"chest pain|heart attack|can'?t breathe|cannot breathe|medical emergency|unconscious|seizure|bleeding|fainted|stranded|unaccompanied|travell?ing alone|child .* missing|harass|assault|unsafe|emergency", 10)],
    "UNRESOLVED_GRIEVANCE": [(r"(second|third|fourth) time|(two|three|four|five|\d+) times", 6), (r"(no ?one|nobody) (has )?(replied|responded)|no response|no reply|still waiting|not resolved|unresolved", 4)],
    "FLIGHT_DISRUPTION": [(r"cancell?ed", 2), (r"delay(ed)?", 2), (r"miss(ed)? (my |the )?connection|connecting flight", 3), (r"re-?book", 3),
                          (r"denied boarding|overbook", 4), (r"compensation", 2), (r"diverted|rescheduled by", 3)],
    "BAGGAGE_ISSUE": [(r"(lost|missing|delayed|damaged|broken|did not arrive|didn'?t arrive|not arrived|never arrived).{0,40}(bag|baggage|luggage|suitcase)"
                       r"|(bag|baggage|luggage|suitcase).{0,40}(lost|missing|delayed|damaged|broken|did not arrive|didn'?t arrive|not arrived|never arrived)", 5),
                      (r"tag number|baggage claim", 2)],
    "REFUND_CHANGES": [(r"refund|money back|reimburse", 4), (r"cancel(l?ation)? (my )?(booking|ticket)", 3),
                       (r"change (my )?(travel )?date|re-?schedul|change fee|name (correction|change)|spelt|spelled|misspel|wrongly", 3)],
    "CHECKIN_SERVICES": [(r"web check-?in|online check-?in|boarding pass|seat (selection|change)|choose (a )?seat|wheelchair|special meal|vegetarian|diabetic meal|assistance", 3), (r"check-?in", 1)],
    "GENERAL_INFO": [(r"allowance|how (much|many|early|long)|what time|timings?|miles|loyalty|policy|counters? close|opens?|closes?", 2)],
}
LEGAL_RE = re.compile(r"legal|lawyer|\bsue\b|court|consumer forum|compensation|social media|media", re.I)


def classify(text):
    low = text.lower()
    scores = {intent: sum(w for pattern, w in rules if re.search(pattern, low)) for intent, rules in INTENT_RULES.items()}
    intent = max(scores, key=lambda i: (scores[i], -list(scores).index(i)))
    if scores[intent] == 0:
        intent = "GENERAL_INFO"
    n_intents = sum(1 for s in scores.values() if s >= 3)           # only clearly present intents count
    if intent == "SAFETY_EMERGENCY":
        priority = "P1"
    elif intent in ("FLIGHT_DISRUPTION", "UNRESOLVED_GRIEVANCE"):
        priority = "P2"
    elif intent == "BAGGAGE_ISSUE":
        priority = "P2" if re.search(r"medic|insulin|prescription", low) else "P3"
    elif intent in ("REFUND_CHANGES", "CHECKIN_SERVICES"):
        priority = "P3"
    else:
        priority = "P4"
    words = len(text.split())
    if LEGAL_RE.search(text) or words > 40 or n_intents >= 3 or intent == "UNRESOLVED_GRIEVANCE":
        complexity = "HIGH"
    elif words > 20 or n_intents == 2:
        complexity = "MEDIUM"
    else:
        complexity = "LOW"
    return {"intent": intent, "priority": priority, "complexity": complexity}


for qid in ("Q001", "Q005", "Q019"):
    rec = next(r for r in valid_preview if r["query_id"] == qid)
    print(qid, classify(rec["text"]))

## 6. Components 3-5 — Knowledge retrieval, resolution score and workflow router

The **resolution score** is `C = 100 x matched / considered`: the share of the query's meaningful (stemmed, stop-word-free)
words that also appear in the best-matching approved article. The router then applies the risk thresholds from the lab.

In [ ]:
STOP = set("the and for you are was were with this that have has had not but can could would should will shall may might from your our any all "
           "its it's i'm i've his her him she they them their there here what when where which who whom how why does did doing done get got "
           "please help need want like also just very much many some more most out off over into onto than then too only own same such "
           "redacted email number".split())


def stem(token):
    for suffix in ("ing", "ed", "es", "s"):
        if token.endswith(suffix) and len(token) - len(suffix) >= 3:
            token = token[: -len(suffix)]
            break
    return token[:-1] if len(token) > 4 and token[-1] == token[-2] else token


def tokens(text):
    return [stem(t) for t in re.findall(r"[a-z]+", text.lower()) if len(t) >= 3 and t not in STOP]


def match_article(text, articles):
    """-> (article, score, matched, considered); score = 100 x matched / considered."""
    query = set(tokens(text))
    best = (None, 0.0, 0, len(query))
    for art in articles:
        keywords = set(tokens(" ".join(art["keywords"] + [art["title"]])))
        matched = len(query & keywords)
        score = round(100 * matched / len(query), 2) if query else 0.0
        if (score, matched) > (best[1], best[2]):
            best = (art, score, matched, len(query))
    return best


def route_ticket(cls, score):
    t = CONFIG["thresholds"]
    if cls["priority"] == "P1":                           return "ESCALATE_IMMEDIATE"      # safety: bypass the bot, no delay
    if cls["intent"] == "UNRESOLVED_GRIEVANCE":           return "ESCALATE_SENIOR"
    if score < t["agent_review"]:                         return "ESCALATE_REQUEST_INFO"
    if cls["priority"] == "P2" or cls["complexity"] == "HIGH":   return "AGENT_REVIEW"         # guardrail: disruptions always see a person
    if score >= t["auto_resolve"] and cls["complexity"] == "LOW": return "AUTO_RESOLVE_DRAFT"
    return "AGENT_REVIEW"


def triage(text, articles):
    cls = classify(text)
    art, score, matched, considered = match_article(text, articles)
    route = route_ticket(cls, score)
    draft = {"ESCALATE_IMMEDIATE": "We have alerted a duty manager, who will contact you immediately. If this is a medical emergency please also tell the nearest crew member or call local emergency services.",
             "ESCALATE_SENIOR": "We are sorry that your earlier messages were not answered. A senior agent now owns your case and will contact you.",
             "ESCALATE_REQUEST_INFO": "Thank you for contacting SkyBridge Airways. To help you we need a few more details: your booking reference, the flight number and a short description of the problem."
             }.get(route, f"Hello, thank you for contacting SkyBridge Airways. {art['answer']}" if art else "")
    return {**cls, "article_id": art["id"] if art else None, "match_score": score, "matched": matched, "considered": considered, "route": route, "draft": draft}


print("Router thresholds:", CONFIG["thresholds"], "| routes: ESCALATE_IMMEDIATE, ESCALATE_SENIOR, ESCALATE_REQUEST_INFO, AGENT_REVIEW, AUTO_RESOLVE_DRAFT")

## 7. Components 6-10 — The support desk

One class ties the pieces together:

* **Ticket lifecycle manager** — an SQLite table and a state machine: `OPEN -> TRIAGED -> ASSIGNED / ESCALATED -> IN_PROGRESS
  -> RESOLVED -> CLOSED` (or `REOPENED`); invalid transitions are refused.
* **Human support module** — `APPROVE, EDIT, HOLD, REQUEST_INFORMATION, ESCALATE, RESOLVE` with role checks (a viewer cannot act;
  P1 tickets need a supervisor). `HOLD` pauses the SLA clock.
* **Notifications** — idempotent, with retries: up to 3 attempts, waiting 1 s then 2 s between them, every attempt logged.
* **SLA monitor** — P1 15 min, P2 60 min, P3 240 min, P4 1440 min; `MET`, `WITHIN_SLA`, `AT_RISK`, `BREACHED`.
* **Audit logger** — hash-chained rows; details are PII-masked before they are written.

In [ ]:
class AuditLog:
    COLS = ["seq", "timestamp", "event", "ticket_id", "actor", "details", "prev_hash", "hash"]

    def __init__(self, path):
        self.path, self.last, self.seq = Path(path), "GENESIS", 0
        with open(self.path, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(self.COLS)

    @staticmethod
    def digest(prev, *fields):
        return hashlib.sha256("|".join([prev, *map(str, fields)]).encode()).hexdigest()

    def append(self, event, ticket_id="-", actor="system", details=""):
        self.seq += 1
        ts, details = datetime.now().isoformat(timespec="seconds"), mask_pii(details)
        h = self.digest(self.last, self.seq, ts, event, ticket_id, actor, details)
        with open(self.path, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([self.seq, ts, event, ticket_id, actor, details, self.last, h])
        self.last = h

    @classmethod
    def verify(cls, path):
        prev = "GENESIS"
        with open(path, newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                if r["prev_hash"] != prev or r["hash"] != cls.digest(prev, r["seq"], r["timestamp"], r["event"], r["ticket_id"], r["actor"], r["details"]):
                    return False, int(r["seq"])
                prev = r["hash"]
        return True, None


TRANSITIONS = {"OPEN": {"TRIAGED"}, "TRIAGED": {"ASSIGNED", "ESCALATED", "IN_PROGRESS"}, "ASSIGNED": {"IN_PROGRESS", "ESCALATED"},
               "ESCALATED": {"IN_PROGRESS", "RESOLVED"}, "IN_PROGRESS": {"RESOLVED", "ON_HOLD", "ESCALATED"}, "ON_HOLD": {"IN_PROGRESS"},
               "RESOLVED": {"CLOSED", "REOPENED"}, "REOPENED": {"IN_PROGRESS", "ESCALATED"}, "CLOSED": set()}
ALL_ACTIONS = {"APPROVE", "EDIT", "HOLD", "REQUEST_INFORMATION", "ESCALATE", "RESOLVE"}
ROLE_ACTIONS = {"agent": ALL_ACTIONS, "supervisor": ALL_ACTIONS, "viewer": set()}


class SupportDesk:
    def __init__(self, workdir, kb_path, sleep=time.sleep):
        self.dir, self.sleep = Path(workdir), sleep
        self.dir.mkdir(parents=True, exist_ok=True)
        self.kb, self.kb_error = load_kb(kb_path)
        self.audit = AuditLog(self.dir / "audit_log.csv")
        self.db = sqlite3.connect(self.dir / "tickets.db")
        self.db.execute("CREATE TABLE tickets (ticket_id TEXT PRIMARY KEY, query_id TEXT, customer_id TEXT, email TEXT, text TEXT, intent TEXT, "
                        "priority TEXT, complexity TEXT, match_score REAL, article_id TEXT, route TEXT, state TEXT, review_status TEXT, created_at TEXT, "
                        "first_response_at TEXT, resolved_at TEXT, assigned_to TEXT, draft TEXT, previous_draft TEXT, hold_started TEXT, hold_seconds REAL DEFAULT 0)")
        with open(self.dir / "notification_log.csv", "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(["timestamp", "ticket_id", "kind", "attempt", "status", "error", "waited_s", "recipient"])
        self.rejected, self.done_keys, self.notify_results, self.log_attempts = [], set(), {}, Counter()

    # -- helpers ---------------------------------------------------------------------------------------
    def row(self, tid):
        cur = self.db.execute("SELECT * FROM tickets WHERE ticket_id=?", (tid,))
        r = cur.fetchone()
        return dict(zip([c[0] for c in cur.description], r)) if r else None

    def tickets(self):
        cur = self.db.execute("SELECT * FROM tickets ORDER BY query_id")
        cols = [c[0] for c in cur.description]
        return [dict(zip(cols, r)) for r in cur.fetchall()]

    def tid(self, query_id):
        return "TKT-" + hashlib.sha1(query_id.encode()).hexdigest()[:8].upper()

    def update(self, tid, **fields):
        self.db.execute(f"UPDATE tickets SET {', '.join(k + '=?' for k in fields)} WHERE ticket_id=?", (*fields.values(), tid))
        self.db.commit()

    def transition(self, tid, new_state, actor="system"):
        old = self.row(tid)["state"]
        if new_state not in TRANSITIONS[old]:
            self.audit.append("INVALID_TRANSITION", tid, actor, f"{old} -> {new_state} refused")
            raise ValueError(f"invalid transition {old} -> {new_state}")
        self.update(tid, state=new_state)
        self.audit.append("STATE_CHANGE", tid, actor, f"{old} -> {new_state}")

    # -- ingestion, triage, ticket creation ---------------------------------------------------------------
    def run(self, raw):
        valid, self.rejected = ingest(raw)
        for r in self.rejected:
            self.audit.append("REJECTED", r["query_id"], "system", f"{r['category']}: {r['reason']}")
        for rec in valid:
            tri = triage(rec["text"], self.kb)
            tid = self.tid(rec["query_id"])
            self.db.execute("INSERT INTO tickets (ticket_id, query_id, customer_id, email, text, intent, priority, complexity, match_score, article_id, route, "
                            "state, review_status, created_at, draft) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                            (tid, rec["query_id"], rec["customer_id"], rec["email"], rec["text"], tri["intent"], tri["priority"], tri["complexity"],
                             tri["match_score"], tri["article_id"], tri["route"], "OPEN", "PENDING_REVIEW", rec["created_at"].isoformat(), tri["draft"]))
            self.db.commit()
            self.audit.append("CLASSIFIED", tid, "system", f"{tri['intent']} {tri['priority']} {tri['complexity']} match {tri['match_score']} -> {tri['route']}")
            self.transition(tid, "TRIAGED")
            self.transition(tid, "ESCALATED" if tri["route"] in ("ESCALATE_IMMEDIATE", "ESCALATE_SENIOR") else "ASSIGNED")
            self.update(tid, assigned_to="supervisor_queue" if tri["route"].startswith("ESCALATE_I") or tri["route"] == "ESCALATE_SENIOR" else "agent_queue")
            self.notify(tid, "RECEIPT", rec["email"], f"We have received your message (ticket {tid}).")
            if tri["priority"] == "P1":
                self.notify(tid, "DUTY_MANAGER_ALERT", "duty.manager@skybridge.example", f"P1 ticket {tid} needs immediate action.")
        return valid

    # -- notifications -------------------------------------------------------------------------------------
    def notify(self, tid, kind, email, body):
        key = f"{tid}:{kind}"
        if key in self.done_keys:                                       # idempotency: the same event is never sent twice
            self.audit.append("NOTIFY_DUPLICATE_SKIPPED", tid, "system", kind)
            return "DUPLICATE_SKIPPED"
        retry, status, waited = CONFIG["retry"], "FAILED", 0.0
        for attempt in range(1, retry["max_attempts"] + 1):
            self.log_attempts[key] += 1
            domain = email.rsplit("@", 1)[-1]
            delivered = domain != "invalid.test" and not (domain == "flaky.example" and attempt < 3)
            with open(self.dir / "notification_log.csv", "a", newline="", encoding="utf-8") as f:
                csv.writer(f).writerow([datetime.now().isoformat(timespec="milliseconds"), tid, kind, attempt, "SENT" if delivered else "FAILED",
                                        "" if delivered else "mailbox unavailable", f"{waited:.1f}", email[:1] + "***@" + domain])
            if delivered:
                status = "SENT"
                break
            if attempt < retry["max_attempts"]:
                waited = retry["base_delay_s"] * retry["factor"] ** (attempt - 1)      # 1 s, then 2 s
                self.sleep(waited)
        self.done_keys.add(key)
        self.notify_results[key] = status
        self.audit.append("NOTIFICATION", tid, "system", f"{kind} {status} after {self.log_attempts[key]} attempt(s)")
        return status

    # -- human support module --------------------------------------------------------------------------------
    def act(self, tid, reviewer, action, text="", at=None):
        at, role, t = at or datetime.now(), CONFIG["reviewers"].get(reviewer), self.row(tid)

        def deny(msg):
            self.audit.append("ACTION_DENIED", tid, reviewer, f"{action}: {msg}")
            return {"ok": False, "message": msg, "status": t["review_status"] if t else None}

        if t is None:                                                   return deny("ticket not found")
        if action not in ROLE_ACTIONS.get(role, set()):                 return deny(f"role '{role}' may not {action}")
        if t["priority"] == "P1" and role != "supervisor":              return deny("P1 tickets must be handled by a supervisor")
        if action in ("EDIT", "HOLD", "REQUEST_INFORMATION", "RESOLVE") and not text.strip(): return deny(f"{action} needs a text / reason")
        if t["state"] == "ON_HOLD":                                     # resuming work ends the hold and releases the SLA clock
            self.update(tid, hold_seconds=(t["hold_seconds"] or 0) + (at - datetime.fromisoformat(t["hold_started"])).total_seconds(), hold_started=None)
        if action != "ESCALATE" and self.row(tid)["state"] != "IN_PROGRESS":
            self.transition(tid, "IN_PROGRESS", reviewer)
        first = t["first_response_at"] or at.isoformat()
        if action == "APPROVE":
            self.notify(tid, "RESPONSE", t["email"], t["draft"])
            self.update(tid, first_response_at=first, review_status="APPROVED_FOR_RESPONSE")
            if t["route"] == "AUTO_RESOLVE_DRAFT":
                self.transition(tid, "RESOLVED", reviewer); self.update(tid, resolved_at=at.isoformat())
        elif action == "EDIT":
            self.update(tid, previous_draft=t["draft"], draft=text, review_status="EDITED")
        elif action == "HOLD":
            self.transition(tid, "ON_HOLD", reviewer); self.update(tid, hold_started=at.isoformat(), review_status="ON_HOLD")
        elif action == "REQUEST_INFORMATION":
            self.notify(tid, "INFO_REQUEST", t["email"], text)
            self.update(tid, first_response_at=first, review_status="INFORMATION_REQUESTED")
        elif action == "ESCALATE":
            self.transition(tid, "ESCALATED", reviewer); self.update(tid, assigned_to="supervisor_queue", review_status="ESCALATED_BY_AGENT")
        elif action == "RESOLVE":
            self.notify(tid, "RESOLUTION", t["email"], text)
            self.transition(tid, "RESOLVED", reviewer); self.update(tid, resolved_at=at.isoformat(), first_response_at=first, review_status="RESOLVED")
        self.audit.append("HUMAN_ACTION", tid, reviewer, f"{action} ({role})" + (f": {text[:40]}" if text else ""))
        return {"ok": True, "message": f"Action '{action}' applied successfully by {reviewer}.", "status": self.row(tid)["review_status"]}

    def reopen(self, tid, actor="customer"):
        self.transition(tid, "REOPENED", actor)
        return self.row(tid)["state"]

    # -- SLA monitor -----------------------------------------------------------------------------------------
    def sla(self, tid, now):
        t = self.row(tid)
        created, limit = datetime.fromisoformat(t["created_at"]), timedelta(minutes=CONFIG["sla_minutes"][t["priority"]])
        paused = timedelta(seconds=t["hold_seconds"] or 0)
        if t["hold_started"]:
            paused += now - datetime.fromisoformat(t["hold_started"])
        deadline = created + limit + paused
        if t["first_response_at"]:
            took = datetime.fromisoformat(t["first_response_at"]) - created
            met = datetime.fromisoformat(t["first_response_at"]) <= deadline
            return {"status": "MET" if met else "BREACHED", "left": 0.0, "detail": f"answered after {took.total_seconds() / 60:.0f} min (limit {int(limit.total_seconds() // 60)})"}
        elapsed = now - created - paused
        fraction = elapsed / limit
        status = "BREACHED" if fraction > 1 else "AT_RISK" if fraction >= CONFIG["at_risk_fraction"] else "WITHIN_SLA"
        left = (deadline - now).total_seconds() / 60
        return {"status": status, "left": left, "detail": f"{left:.0f} min left" if left >= 0 else f"{-left:.0f} min overdue"}


print("SupportDesk ready: ingestion, triage, SQLite tickets, human actions, notifications with retry, SLA monitor, audit log")

## 8. Running the pipeline

Two customers have broken mailboxes on purpose, so the notification retries run with their **real 1 s and 2 s waits**.

In [ ]:
desk = SupportDesk(WORK, WORK / "knowledge_base.json")
t0 = time.perf_counter()
valid = desk.run(raw_rows)
run_seconds = time.perf_counter() - t0

print(f"python app.py  ->  {len(raw_rows)} queries ingested | {len(valid)} tickets created | {len(desk.rejected)} rejected | {run_seconds:.1f} s "
      f"(includes the notification back-off waits)\n")
print(f"{'Query':<6}{'Intent':<22}{'Prio':<5}{'Cmplx':<7}{'Match':>6}  {'Art':<5}{'Route':<23}{'Ticket'}")
for t in desk.tickets():
    print(f"{t['query_id']:<6}{t['intent']:<22}{t['priority']:<5}{t['complexity']:<7}{t['match_score']:>6.1f}  {t['article_id'] or '-':<5}{t['route']:<23}{t['ticket_id']}")
routes = Counter(t["route"] for t in desk.tickets())
print("\nRoutes:", ", ".join(f"{k} {v}" for k, v in routes.most_common()))

## 9. Component 6 — Ticket lifecycle (state machine and SQLite)

A separate sandbox desk shows a ticket travelling through every state, a refused invalid transition and a reopened ticket.

In [ ]:
sandbox = SupportDesk(WORK / "sandbox", WORK / "knowledge_base.json", sleep=lambda s: None)
sandbox.run([r for r in raw_rows if r["query_id"] == "Q001"])
tid = sandbox.tid("Q001")
steps = [sandbox.row(tid)["state"]]
sandbox.act(tid, "AGT_01", "APPROVE", at=datetime(2026, 11, 10, 10, 0))
steps.append(sandbox.row(tid)["state"])
sandbox.transition(tid, "CLOSED")
steps.append(sandbox.row(tid)["state"])
print(f"LIFECYCLE of {tid} (Q001, auto-resolve route):  OPEN -> TRIAGED -> ASSIGNED -> IN_PROGRESS -> RESOLVED -> CLOSED")
print("  state after approval:", steps[1], "| after closing:", steps[2])
try:
    sandbox.transition(tid, "IN_PROGRESS")
except ValueError as exc:
    print(f"  invalid move refused: {exc}")
sandbox.run([r for r in raw_rows if r["query_id"] == "Q002"])
t2 = sandbox.tid("Q002")
sandbox.act(t2, "AGT_01", "RESOLVE", "web check-in times explained", at=datetime(2026, 11, 10, 10, 0))
print(f"  customer replies to {t2}: {sandbox.row(t2)['state']} -> {sandbox.reopen(t2)}")

counts = pd.read_sql("SELECT state, COUNT(*) AS tickets FROM tickets GROUP BY state ORDER BY tickets DESC", desk.db)
print("\ntickets.db after triage (main desk):\n" + counts.to_string(index=False))
sample = pd.read_sql("SELECT ticket_id, query_id, priority, state, assigned_to FROM tickets WHERE query_id IN ('Q001','Q005','Q008','Q015')", desk.db)
print("\nsample rows:\n" + sample.to_string(index=False))

## 10. Component 7 — Human-in-the-loop support console

The console lists tickets that need a person, ordered by priority and shows SLA time left. Keystrokes are scripted here
(set `INTERACTIVE_HITL = True` to type your own). Afterwards a batch of further decisions is applied quietly; every action is
audited. Reviewer response times are **simulated** minutes after each ticket was created.

In [ ]:
MENU = "Select Action:\n  1. APPROVE (dispatch the proposed response)    2. EDIT (change the draft)    3. HOLD (pause the SLA clock)\n  4. REQUEST_INFORMATION   5. ESCALATE   6. RESOLVE"
ACTION_BY_KEY = {"1": "APPROVE", "2": "EDIT", "3": "HOLD", "4": "REQUEST_INFORMATION", "5": "ESCALATE", "6": "RESOLVE"}
created = lambda qid: datetime.fromisoformat(desk.row(desk.tid(qid))["created_at"])


def hitl_console(queue, script, show=5):
    print("=" * 74 + "\n  HUMAN-IN-THE-LOOP (HITL) SUPPORT CONSOLE\n" + "=" * 74)
    print(f"\n{len(queue)} tickets are waiting for a person (P1 first):\n")
    for i, qid in enumerate(queue[:show], 1):
        t, s = desk.row(desk.tid(qid)), desk.sla(desk.tid(qid), CONSOLE_TIME)
        print(f"[{i}] {t['ticket_id']} | Priority: {t['priority']} | State: {t['state']} | SLA: {s['status']} ({s['detail']})")
        print(f"    Intent: {t['intent']} | Route: {t['route']} | Customer: {t['customer_id']}")
    print()
    for reviewer, qid, key, text, minutes in script:
        tid, t = desk.tid(qid), desk.row(desk.tid(qid))
        print(f"-- {reviewer} opens ticket {tid} ({qid}, {t['priority']})\n   Summary: {t['text'][:76]}")
        print("   " + MENU.replace("\n", "\n   "))
        choice = input("   > ") if INTERACTIVE_HITL else key
        print(f"   > {choice}" if not INTERACTIVE_HITL else "")
        res = desk.act(tid, reviewer, ACTION_BY_KEY[choice], text, at=created(qid) + timedelta(minutes=minutes))
        print(f"   [{'SUCCESS' if res['ok'] else 'DENIED'}] {res['message']}  Status: {res['status']}\n")


waiting = sorted([t for t in desk.tickets() if t["route"] != "AUTO_RESOLVE_DRAFT"], key=lambda t: (t["priority"], t["created_at"]))
hitl_console([t["query_id"] for t in waiting], [
    ("OBS_01", "Q005", "1", "", 5),                                    # a read-only observer tries to approve
    ("AGT_01", "Q005", "1", "", 6),                                    # an agent tries to take a P1 ticket
    ("SUP_01", "Q005", "1", "", 9),                                    # the supervisor acts within the 15-minute SLA
    ("AGT_01", "Q004", "2", "We have rebooked you on the next flight; please confirm the 14:20 departure.", 32),
    ("AGT_01", "Q004", "1", "", 38)])

In [ ]:
bulk = [("SUP_01", "Q017", "APPROVE", "", 12), ("SUP_01", "Q008", "APPROVE", "", 14), ("AGT_02", "Q019", "APPROVE", "", 75),
        ("AGT_01", "Q001", "APPROVE", "", 1), ("AGT_01", "Q002", "APPROVE", "", 2), ("AGT_01", "Q003", "APPROVE", "", 3),
        ("AGT_01", "Q006", "APPROVE", "", 20), ("AGT_01", "Q009", "APPROVE", "", 25), ("AGT_01", "Q010", "APPROVE", "", 30),
        ("AGT_01", "Q014", "APPROVE", "", 28), ("AGT_01", "Q018", "APPROVE", "", 40),
        ("AGT_01", "Q016", "REQUEST_INFORMATION", "Please send a copy of the passport page so we can verify the spelling.", 33),
        ("AGT_02", "Q020", "HOLD", "waiting for catering to confirm meal availability", 10)]
applied = Counter()
for reviewer, qid, action, text, minutes in bulk:
    res = desk.act(desk.tid(qid), reviewer, action, text, at=created(qid) + timedelta(minutes=minutes))
    applied[action] += res["ok"]
print("Batch decisions applied:", dict(applied), "| tickets still untouched (Q007, Q015) are left to show the SLA monitor")

## 11. Component 9 — SLA monitor

In [ ]:
print(f"SLA BOARD at {NOW:%Y-%m-%d %H:%M}  (P1 15 min | P2 60 min | P3 240 min | P4 1440 min, clock paused while a ticket is on hold)\n")
print(f"{'Ticket':<14}{'Query':<6}{'Prio':<5}{'State':<12}{'SLA':<12}Detail")
sla_rows = []
for t in sorted(desk.tickets(), key=lambda t: (t["priority"], t["query_id"])):
    s = desk.sla(t["ticket_id"], NOW)
    sla_rows.append(s["status"])
    print(f"{t['ticket_id']:<14}{t['query_id']:<6}{t['priority']:<5}{t['state']:<12}{s['status']:<12}{s['detail']}")
print("\nSummary:", dict(Counter(sla_rows)))

## 12. Component 10 — Audit log (hash chain, PII masking) and Component 8 — notification retry log

In [ ]:
ok, bad = AuditLog.verify(desk.dir / "audit_log.csv")
audit = pd.read_csv(desk.dir / "audit_log.csv", dtype=str, keep_default_na=False)
print(f"audit_log.csv: {len(audit)} rows | hash chain intact: {ok} | events: " + ", ".join(f"{k} {v}" for k, v in Counter(audit['event']).most_common(5)))
pick = audit[audit["event"].isin(["REJECTED", "CLASSIFIED", "ACTION_DENIED", "HUMAN_ACTION", "NOTIFICATION"])]
pick = pd.concat([pick[pick.event == "REJECTED"].head(2), pick[pick.event == "CLASSIFIED"].iloc[[4, 13]], pick[pick.event == "ACTION_DENIED"].head(2), pick[pick.event == "HUMAN_ACTION"].head(2)])
print(f"\n{'seq':<5}{'event':<14}{'ticket':<14}{'actor':<8}{'details':<62}hash")
for _, r in pick.iterrows():
    print(f"#{r['seq']:<4}{r['event']:<14}{r['ticket_id']:<14}{r['actor']:<8}{r['details'][:60]:<62}{r['hash'][:8]}")
text = (desk.dir / "audit_log.csv").read_text(encoding="utf-8")
print("\nPII check on the audit log:  raw phone present =", "9876543210" in text, "| raw booking ref present =", "K7P2QZ" in text,
      "| raw e-mail present =", bool(EMAIL_RE.search(text)))
broken = text.splitlines()
row_no = next(i for i, line in enumerate(broken) if i > 20 and "CLASSIFIED" in line)
broken[row_no] = broken[row_no].replace("CLASSIFIED", "CLASSIFIEX", 1)
(desk.dir / "audit_tampered.csv").write_text("\n".join(broken) + "\n", encoding="utf-8")
print("Tamper test: one historic row edited ->", "chain intact:", AuditLog.verify(desk.dir / "audit_tampered.csv")[0])

In [ ]:
log_df = pd.read_csv(desk.dir / "notification_log.csv", dtype=str, keep_default_na=False)
print(f"notification_log.csv: {len(log_df)} attempts for {log_df[['ticket_id', 'kind']].drop_duplicates().shape[0]} notifications (recipients masked)\n")
print("Retry evidence: exponential back-off (1 s, then 2 s) between the three attempts")
shown = log_df[log_df["ticket_id"].isin([desk.tid("Q006"), desk.tid("Q008")]) & log_df["kind"].isin(["RECEIPT"])]
print(shown[["timestamp", "ticket_id", "kind", "attempt", "status", "error", "waited_s", "recipient"]].to_string(index=False))
final = Counter(desk.notify_results.values())
print(f"\nFinal outcome per notification: {final['SENT']} SENT, {final['FAILED']} FAILED (FAILED ones are kept in the log for manual retry)")

## 13. MCP-style tool interface and test suite TC01-TC22

A tiny tool registry exposes two tools to *external AI agents* — `triage_query` and `get_sla_status` — with role checks. Then 22 test cases
run on isolated desks (retry waits disabled for speed).

In [ ]:
class ToolRegistry:
    """MCP-style: register tools once, let each role discover only what it may call, authorize every call."""
    def __init__(self):
        self.tools, self.denied = {}, []

    def register(self, name, func, description, roles):
        self.tools[name] = (func, description, set(roles))

    def discover(self, role):
        return {n: d for n, (f, d, r) in self.tools.items() if role in r}

    def call(self, role, name, **kwargs):
        if name not in self.tools:
            raise LookupError(f"unknown tool '{name}'")
        func, _, roles = self.tools[name]
        if role not in roles:
            self.denied.append((role, name))
            raise PermissionError(f"role '{role}' may not call '{name}'")
        return func(**kwargs)


TOOLS = ToolRegistry()
TOOLS.register("triage_query", lambda text: {k: v for k, v in triage(mask_pii(clean_input(text)), kb).items() if k in ("intent", "priority", "complexity", "route")},
               "Classify a query and return the proposed route", {"external_agent"})
TOOLS.register("get_sla_status", lambda ticket_id: desk.sla(ticket_id, NOW), "Return live SLA status of a ticket", {"external_agent", "viewer"})
print("MCP-style tools for external_agent:", list(TOOLS.discover("external_agent")), "| for viewer:", list(TOOLS.discover("viewer")))

In [ ]:
TESTS = []
fresh = lambda name, **kw: SupportDesk(WORK / "tests" / name, WORK / "knowledge_base.json", sleep=lambda s: None, **kw)


def case(tc, scenario, expected, ok):
    TESTS.append((tc, scenario, expected, "PASS" if ok else "FAIL"))


def one(text, qid="T01", created="2026-11-10T09:00:00", email="t01@mail.example"):
    return [{"query_id": qid, "customer_id": "C900", "customer_email": email, "query_text": text, "created_at": created}]


def run_one(name, text, **kw):
    d = fresh(name)
    d.run(one(text, **kw))
    return d, d.row(d.tid(kw.get("qid", "T01")))


d, t = run_one("tc01", "What is the checked baggage allowance for economy class?");       case("TC01", "Valid information query", "GENERAL_INFO / P4", (t["intent"], t["priority"]) == ("GENERAL_INFO", "P4"))
d, t = run_one("tc02", "I need a refund for my cancelled flight SB118, booked with card"); case("TC02", "Refund request", "REFUND_CHANGES / P3", (t["intent"], t["priority"]) == ("REFUND_CHANGES", "P3"))
d, t = run_one("tc03", "I am feeling chest pain and I am on flight SB512 right now");     case("TC03", "Medical emergency on board", "P1 + immediate escalation", (t["priority"], t["route"]) == ("P1", "ESCALATE_IMMEDIATE"))
d, t = run_one("tc04", "My flight got cancelled and I will miss my connection, please rebook me"); case("TC04", "Cancellation with missed connection", "FLIGHT_DISRUPTION / P2", (t["intent"], t["priority"]) == ("FLIGHT_DISRUPTION", "P2"))
d = fresh("tc05"); d.run(one("   ")); case("TC05", "Empty query text", "Rejected: EMPTY_QUERY", [r["category"] for r in d.rejected] == ["EMPTY_QUERY"])
d = fresh("tc06"); d.run(one("Where is my refund?") + one("Where is my bag?")); case("TC06", "Duplicate query ID", "Flagged DUPLICATE_ID, one ticket", [r["category"] for r in d.rejected] == ["DUPLICATE_ID"] and len(d.tickets()) == 1)
d, t = run_one("tc07", "Can I bring my pet python on board?");                            case("TC07", "No matching knowledge article", "Escalate + request information", t["route"] == "ESCALATE_REQUEST_INFO" and t["match_score"] < 40)
d, t = run_one("tc08", "checked baggage allowance weight limit");                          case("TC08", "Match >= 70, low complexity", "AUTO_RESOLVE_DRAFT", t["route"] == "AUTO_RESOLVE_DRAFT" and t["match_score"] >= 70)
d, t = run_one("tc09", "How much luggage weight can I carry?");                            case("TC09", "Match 40-69.99", "AGENT_REVIEW", t["route"] == "AGENT_REVIEW" and 40 <= t["match_score"] < 70)
d, t = run_one("tc10", "I have chest pain, what is the checked baggage allowance weight limit?"); case("TC10", "P1 query with a high knowledge match", "Escalation takes precedence", t["route"] == "ESCALATE_IMMEDIATE" and t["match_score"] >= 40)
d = fresh("tc11"); d.run(one("What is the checked baggage allowance?")); case("TC11", "Notification succeeds first time", "Status SENT, 1 attempt", d.notify_results[d.tid("T01") + ":RECEIPT"] == "SENT" and d.log_attempts[d.tid("T01") + ":RECEIPT"] == 1)
d = fresh("tc12"); d.run(one("What is the checked baggage allowance?", email="x@flaky.example")); k = d.tid("T01") + ":RECEIPT"; case("TC12", "Delivery fails twice, then succeeds", "3 attempts recorded, SENT", d.log_attempts[k] == 3 and d.notify_results[k] == "SENT")
d = fresh("tc13"); d.run(one("What is the checked baggage allowance?", email="x@invalid.test")); k = d.tid("T01") + ":RECEIPT"; case("TC13", "All delivery attempts fail", "Status FAILED, 3 attempts", d.log_attempts[k] == 3 and d.notify_results[k] == "FAILED")
d, t = run_one("tc14", "Please change my travel date for my booking", qid="T14", created="2026-11-10T06:30:00"); case("TC14", "Ticket approaches SLA deadline", "AT_RISK warning", d.sla(t["ticket_id"], NOW)["status"] == "AT_RISK")
d, t = run_one("tc15", "My flight was delayed and I want to be rebooked", qid="T15", created="2026-11-10T08:00:00"); case("TC15", "Ticket exceeds SLA deadline", "BREACHED recorded", d.sla(t["ticket_id"], NOW)["status"] == "BREACHED")
d, t = run_one("tc16", "How much luggage weight can I carry?", qid="T16"); d.act(t["ticket_id"], "AGT_01", "EDIT", "Corrected draft text."); t2 = d.row(t["ticket_id"]); case("TC16", "Agent edits a response", "Previous draft saved", bool(t2["previous_draft"]) and t2["draft"] == "Corrected draft text.")
d, t = run_one("tc17", "How much luggage weight can I carry?", qid="T17"); a = d.act(t["ticket_id"], "OBS_01", "APPROVE"); case("TC17", "Unauthorized approval attempt", "Permission denied and logged", not a["ok"] and "ACTION_DENIED" in (d.dir / "audit_log.csv").read_text())
d, t = run_one("tc18", "How much luggage weight can I carry?", qid="T18"); d.act(t["ticket_id"], "AGT_01", "RESOLVE", "answered"); d.reopen(t["ticket_id"]); case("TC18", "Resolved ticket reopened by customer", "State -> REOPENED", d.row(t["ticket_id"])["state"] == "REOPENED")
Path(WORK / "bad_kb.json").write_text("{ this is not valid json"); bad = SupportDesk(WORK / "tests" / "tc19", WORK / "bad_kb.json", sleep=lambda s: None); bad.run(one("What is the checked baggage allowance?")); case("TC19", "Malformed knowledge-base JSON", "Handled gracefully, escalates", bad.kb_error is not None and bad.tickets()[0]["route"] == "ESCALATE_REQUEST_INFO")
d = fresh("tc20"); d.run(one("What is the checked baggage allowance?")); tid20 = d.tid("T01"); again = d.notify(tid20, "RECEIPT", "t01@mail.example", "x"); case("TC20", "Duplicate notification event", "Idempotency prevents resend", again == "DUPLICATE_SKIPPED" and d.log_attempts[tid20 + ":RECEIPT"] == 1)
reply = TOOLS.call("external_agent", "triage_query", text="I am having chest pain"); denied = False
try: TOOLS.call("viewer", "triage_query", text="hello")
except PermissionError: denied = True
case("TC21", "MCP-style triage tool and live SLA retrieval", "P1 escalate; viewer denied", reply["priority"] == "P1" and denied and "status" in TOOLS.call("viewer", "get_sla_status", ticket_id=desk.tid("Q007")))
d = fresh("tc22"); d.run(one("<script>alert(1)</script> Call me on 9876543210, PNR K7P2QZ, mail me at jo@mail.example about my bag allowance")); stored = d.tickets()[0]["text"]; audit_text = (d.dir / "audit_log.csv").read_text()
case("TC22", "Script tag and personal data in a query", "Sanitized, PII masked everywhere", "<script" not in stored and not any(x in stored + audit_text for x in ("9876543210", "K7P2QZ", "jo@mail.example")))

print("=" * 90 + "\n" + "LABORATORY TEST EXECUTION SUMMARY TABLE".center(90) + "\n" + "=" * 90)
print(f"{'Test ID':<8}| {'Test scenario':<44}| {'Expected result':<34}| Status")
print("-" * 90)
for tc, scenario, expected, status in TESTS:
    print(f"{tc:<8}| {scenario:<44}| {expected:<34}| {status}")
failures = sum(t[3] == "FAIL" for t in TESTS)
print("=" * 90 + f"\nTotal Tests Run: {len(TESTS)} | Failures: {failures} | Errors: 0\nOverall Result: {'ALL TESTS PASSED' if failures == 0 else 'SOME TESTS FAILED'}\n" + "=" * 90)

## 14. Component 11 — Quantitative evaluation metrics worksheet

In [ ]:
tickets = desk.tickets()
n_rows, n_valid = len(raw_rows), len(tickets)
both = sum((t["intent"], t["priority"]) == GROUND_TRUTH[t["query_id"]] for t in tickets)
intent_ok = sum(t["intent"] == GROUND_TRUTH[t["query_id"]][0] for t in tickets)
prio_ok = sum(t["priority"] == GROUND_TRUTH[t["query_id"]][1] for t in tickets)
auto = sum(t["route"] == "AUTO_RESOLVE_DRAFT" for t in tickets)
eligible = sum(t["priority"] != "P1" and t["intent"] != "UNRESOLVED_GRIEVANCE" for t in tickets)
escalated = sum(t["route"].startswith("ESCALATE") for t in tickets)
delivered, requests_ = sum(v == "SENT" for v in desk.notify_results.values()), len(desk.notify_results)
statuses = [desk.sla(t["ticket_id"], NOW)["status"] for t in tickets]
sla_met, sla_eligible = statuses.count("MET"), sum(s in ("MET", "BREACHED") for s in statuses)
pct = lambda a, b: f"{100 * a / b:.2f}%"

print("QUANTITATIVE EVALUATION WORKSHEET\n")
rows_ = [("1. Validation success rate", "valid / total ingested x 100", f"{n_valid}/{n_rows}", pct(n_valid, n_rows)),
         ("2. Classification accuracy (intent AND priority)", "correct / total classified x 100", f"{both}/{n_valid}", pct(both, n_valid)),
         ("   intent only", "", f"{intent_ok}/{n_valid}", pct(intent_ok, n_valid)),
         ("   priority only", "", f"{prio_ok}/{n_valid}", pct(prio_ok, n_valid)),
         ("3. Knowledge-base auto-resolution rate", "auto-resolved / eligible queries x 100", f"{auto}/{eligible}", pct(auto, eligible)),
         ("4. Escalation rate", "escalated to humans / valid x 100", f"{escalated}/{n_valid}", pct(escalated, n_valid)),
         ("5. Notification delivery success rate", "delivered / delivery requests x 100", f"{delivered}/{requests_}", pct(delivered, requests_)),
         ("6. SLA compliance rate", "met first-response SLA / determinable x 100", f"{sla_met}/{sla_eligible}", pct(sla_met, sla_eligible))]
print(f"{'Metric':<50}{'Formula':<46}{'Counts':<14}Value")
for name, formula, counts_, value in rows_:
    print(f"{name:<50}{formula:<46}{counts_:<14}{value}")
wrong = [(t["query_id"], t["intent"], t["priority"], GROUND_TRUTH[t["query_id"]]) for t in tickets if (t["intent"], t["priority"]) != GROUND_TRUTH[t["query_id"]]]
print("\nMisclassified:", wrong or "none")
print("(eligible = valid queries that are neither P1 safety cases nor unresolved grievances; SLA 'determinable' = answered, or deadline already passed)")
summary = {"rows": n_rows, "valid": n_valid, "rejected": len(desk.rejected), "accuracy_both": [both, n_valid], "intent": [intent_ok, n_valid], "priority": [prio_ok, n_valid],
           "auto": [auto, eligible], "escalated": [escalated, n_valid], "notifications": [delivered, requests_], "sla": [sla_met, sla_eligible],
           "tests": [sum(t[3] == "PASS" for t in TESTS), len(TESTS)], "routes": dict(routes), "run_seconds": round(run_seconds, 1),
           "sla_summary": dict(Counter(statuses)), "misclassified": [list(map(str, w)) for w in wrong]}
(WORK / "experiment_08_summary.json").write_text(json.dumps(summary, indent=2))
print("\nArtifacts:", ", ".join(sorted(p.name for p in WORK.iterdir() if p.is_file())))

## Conclusion

The desk ingested and sanitized synthetic passenger queries, classified each by intent, complexity and priority, matched them against an
approved knowledge base and routed them by risk: safety cases went straight to a supervisor, low-risk questions got a draft answer,
unclear ones went to agents or were escalated for more information. Tickets moved through a controlled lifecycle in SQLite, humans
approved, edited, held or resolved them under role checks, notifications were retried with back-off, the SLA monitor flagged at-risk and
breached tickets, and a hash-chained, PII-masked audit log recorded everything. The test suite and the metric worksheet summarise the result.